# Task
Analyse the provided transaction dataset in Python and perform a structured data quality audit.

# What you should do
- Briefly describe the dataset and its possible business use case.
- Identify data quality issues and map them to relevant dimensions such as completeness, uniqueness, validity, consistency, and integrity.
- Compute at least 3 KPIs such as Completeness Rate, Error Rate, or Duplication Rate.
- Define at least 3 validation rules to detect problematic records automatically.
- Prepare a short audit summary with issue type, affected rows, severity, and recommended next action.
- Suggest suitable cleaning steps for the next chapter, but do not implement full cleaning yet.

# Submit
- Completed notebook
- KPI summary and short interpretation
- Validation rules and audit findings
- Recommended cleaning actions for next week

In [1]:
import pandas as pd

# Task 1: Dataset description
Briefly describe the dataset and its possible business use case.

In [2]:
df = pd.read_csv("../../data/week2_customer_transactions_messy.csv")
print('Shape:', df.shape)
df.head()

Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


## Dataset Description
The dataset contains 11 transaction records across 9 columns: transaction_id, customer_id, transaction_date, amount, currency, payment_method, status, region, and last_updated. Each row represents a single customer payment transaction. Transactions span four regions (DE, FR, US, NL), two currencies (EUR, USD), three payment methods (card, bank_transfer, cash), and three status values (completed, pending, cancelled). The dataset contains a number of data quality issues across multiple dimensions, which are identified and analysed in Task 2.

## Business use case
A payments or finance team could use this data to monitor transaction volumes, track payment method adoption, and audit transaction statuses across regions. The status column (completed, pending, cancelled) alongside amount and region could feed operational dashboards or fraud detection pipelines.

# Task 2: Issues by dimension
Identify data quality issues and map them to relevant dimensions such as completeness, uniqueness, validity, consistency, and integrity.

In [3]:
# Count missing values per column
missing_counts = df.isna().sum().rename('missing_count').to_frame()
missing_counts['missing_%'] = (missing_counts['missing_count'] / len(df) * 100).round(1)
display(missing_counts)

# Flagged rows
completeness_mask = df.isna().any(axis=1)
if completeness_mask.any():
    print("\nRows with at least one missing value")
    display(df[completeness_mask])

,missing_count,missing_%
transaction_id,0,0.0
customer_id,1,9.1
transaction_date,0,0.0
amount,1,9.1
currency,0,0.0
payment_method,1,9.1
status,0,0.0
region,0,0.0
last_updated,1,9.1



Rows with at least one missing value


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
3,T0004,NaN,2026-01-07,250.0,EUR,card,pending,FR,2026-01-08
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN
10,T0010,C109,2026-01-11,52.1,EUR,NaN,completed,NL,2026-01-11


In [4]:
# Count duplicate values per column on relevant columns
dup_mask = df.duplicated(subset=['transaction_id'], keep=False)

uniqueness_summary = pd.DataFrame({
    'duplicate_count': {'transaction_id': int(dup_mask.sum())}
})
display(uniqueness_summary)

if dup_mask.any():
    print("\nDuplicate transaction_id")
    display(df[dup_mask])

,duplicate_count
transaction_id,2



Duplicate transaction_id


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15


In [5]:
# Each column checked against its own rules where applicable
valid_currencies = {'EUR', 'USD', 'GBP'}
valid_payment_methods = {'card', 'bank_transfer', 'cash'}
valid_statuses = {'completed', 'pending', 'cancelled'}
valid_regions = {'DE', 'FR', 'US', 'NL', 'GB'}

parsed_dates = pd.to_datetime(df['transaction_date'], errors='coerce', format='%Y-%m-%d')
correct_format = df['transaction_date'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
amount_numeric = pd.to_numeric(df['amount'], errors='coerce')

validation_rules = {
    'transaction_date: impossible date': correct_format & parsed_dates.isna(),
    'amount: negative or zero': amount_numeric <= 0,
    'amount: outlier (> 100,000)': amount_numeric > 100_000,
    'currency: not in whitelist': ~df['currency'].str.upper().isin(valid_currencies),
    'payment_method: not in whitelist': df['payment_method'].notna() & ~df['payment_method'].str.lower().isin(
        valid_payment_methods),
    'status: not in whitelist': df['status'].notna() & ~df['status'].str.lower().isin(valid_statuses),
    'region: not in whitelist': df['region'].notna() & ~df['region'].str.upper().isin(valid_regions),
}

validity_summary = pd.DataFrame({
    'violation_count': {rule: int(mask.sum()) for rule, mask in validation_rules.items()}
})
display(validity_summary)

for rule, mask in validation_rules.items():
    if mask.any():
        print(f"\nViolating {rule}")
        display(df[mask])

,violation_count
transaction_date: impossible date,1
amount: negative or zero,2
"amount: outlier (> 100,000)",1
currency: not in whitelist,1
payment_method: not in whitelist,0
status: not in whitelist,0
region: not in whitelist,0



Violating transaction_date: impossible date


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
7,T0007,C106,2026-02-30,47.0,EUR,card,completed,DE,2026-02-15



Violating amount: negative or zero


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
1,T0002,C101,2026/01/06,0.0,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.0,USD,bank_transfer,completed,US,2026-01-07



Violating amount: outlier (> 100,000)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
8,T0008,C107,2026-01-10,1000000.0,EUR,card,completed,DE,2026-01-10



Violating currency: not in whitelist


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


In [6]:
# Check for formatting inconsistencies (valid value, wrong form)
std_date_format = df['transaction_date'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)

consistency_rules = {
    'transaction_date: non-standard format': ~std_date_format,
    'currency: not uppercase': df['currency'].notna() & (df['currency'] != df['currency'].str.upper()),
    'payment_method: not lowercase': df['payment_method'].notna() & (
                df['payment_method'] != df['payment_method'].str.lower()),
    'status: not lowercase': df['status'].notna() & (df['status'] != df['status'].str.lower()),
    'region: not uppercase': df['region'].notna() & (df['region'] != df['region'].str.upper()),
}

consistency_summary = pd.DataFrame({
    'violation_count': {rule: int(mask.sum()) for rule, mask in consistency_rules.items()}
})
display(consistency_summary)

for rule, mask in consistency_rules.items():
    if mask.any():
        print(f"\nViolating {rule}")
        display(df[mask])

,violation_count
transaction_date: non-standard format,2
currency: not uppercase,0
payment_method: not lowercase,1
status: not lowercase,0
region: not uppercase,1



Violating transaction_date: non-standard format


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
1,T0002,C101,2026/01/06,0.0,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.0,USD,bank_transfer,completed,US,2026-01-07



Violating payment_method: not lowercase


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
1,T0002,C101,2026/01/06,0.0,EUR,CARD,completed,de,2026-01-20



Violating region: not uppercase


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
1,T0002,C101,2026/01/06,0.0,EUR,CARD,completed,de,2026-01-20


In [7]:
# Referential and temporal relationship checks
# last_updated before transaction_date is only checked where transaction_date parsed successfully

last_updated_parsed = pd.to_datetime(df['last_updated'], errors='coerce')
transaction_date_parsed = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
date_parsed_ok = transaction_date_parsed.notna()

integrity_rules = {
    'customer_id: missing (unattributed transaction)': df['customer_id'].isna(),
    'last_updated: precedes transaction_date': (last_updated_parsed < transaction_date_parsed) & date_parsed_ok,
}

integrity_summary = pd.DataFrame({
    'violation_count': {rule: int(mask.sum()) for rule, mask in integrity_rules.items()}
})
display(integrity_summary)

for rule, mask in integrity_rules.items():
    if mask.any():
        print(f"\nViolating {rule}")
        display(df[mask])

,violation_count
customer_id: missing (unattributed transaction),1
last_updated: precedes transaction_date,1



Violating customer_id: missing (unattributed transaction)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
3,T0004,NaN,2026-01-07,250.0,EUR,card,pending,FR,2026-01-08



Violating last_updated: precedes transaction_date


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
2,T0003,C102,06-01-2026,-35.0,USD,bank_transfer,completed,US,2026-01-07


In [8]:
# Internal update lag: how many days between transaction_date and last_updated
# Only checked where both dates parsed successfully
# Rows where lag exceeds 7 days are flagged

valid_dates = transaction_date_parsed.notna() & last_updated_parsed.notna()
lag = (last_updated_parsed - transaction_date_parsed).dt.days
timeliness_mask = valid_dates & (lag > 7)

timeliness_summary = pd.DataFrame({
    'violation_count': {'last_updated lag > 7 days': int(timeliness_mask.sum())}
})
display(timeliness_summary)

if timeliness_mask.any():
    print("\nViolating last_updated lag > 7 days")
    display(df[timeliness_mask])

,violation_count
last_updated lag > 7 days,3



Violating last_updated lag > 7 days


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15


Each column was assessed against the six core data quality dimensions of completeness,
uniqueness, accuracy, consistency, integrity, and timeliness where the check was
meaningful given the available data. Validity, an advanced dimension, was also partially
evaluated using domain constraints.

Completeness was checked across all nine columns. Four cells are missing across three
records: `customer_id` (T0004), `amount` (T0009), `payment_method` (T0010), and
`last_updated` (T0009), each at 9.1% missing. Uniqueness was restricted to
`transaction_id` as the sole primary identifier; all other columns are expected to
contain repeated values. The duplication check uses `keep=False`, flagging both copies
of a duplicate pair rather than only the second occurrence, since it cannot be assumed
that the first record is the correct one. Two records share the transaction_id T0006,
constituting a full duplicate. Validity rules were applied using value whitelists for
`currency`, `payment_method`, `status`, and `region`, an impossible calendar date check
for `transaction_date`, and numeric bounds for `amount`. Four validity violations were
found: one impossible date (2026-02-30), two amount violations (one zero, one negative),
one amount outlier (1,000,000), and one non-standard currency value (EURO).
`payment_method`, `status`, and `region` all passed their whitelist checks. Consistency
checks flagged deviations from a standard form within the dataset: two non-standard date
formats, one uppercase `payment_method` (CARD), and one lowercase `region` (de).
Integrity checks covered the referential dependency of `transaction_id` on `customer_id`
and the temporal constraint that `last_updated` must not precede `transaction_date`; one
violation was found for each. For timeliness, external timeliness cannot be assessed from
a static file; internal update lag between `transaction_date` and `last_updated` is used
as a proxy, with three records exceeding the seven-day threshold. Records where
`last_updated` precedes `transaction_date` are treated as integrity violations rather than
timeliness violations, as they indicate a structural data error rather than a delay.

Accuracy requires comparison against an authoritative external source such as original
transaction logs or a source system of record and cannot be performed on a static file;
no accuracy check is carried out in this audit. Of the three advanced dimensions, only
validity can be partially evaluated here using domain constraints; a complete assessment
would require full knowledge of all business rules governing the data. Relevance depends
on the specific downstream use case and cannot be determined from the dataset alone.
Traceability requires access to upstream pipeline metadata, transformation logs, and
source system documentation, none of which are available here.

# Task 3: KPIs
Compute at least 3 KPIs such as Completeness Rate, Error Rate, or Duplication Rate.

In [9]:
total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isna().sum().sum()
completeness_rate = 1 - (missing_cells / total_cells)

print(f"Completeness Rate: {completeness_rate:.2%}")

Completeness Rate: 95.96%


In [10]:
duplication_rate = dup_mask.sum() / len(df)

print(f"Duplication Rate: {duplication_rate:.2%}")

Duplication Rate: 18.18%


In [11]:
validity_error_rate = pd.DataFrame(validation_rules).any(axis=1).mean()
print(f"Validity Error Rate: {validity_error_rate:.2%}")

Validity Error Rate: 45.45%


In [12]:
integrity_violation_rate = pd.DataFrame(integrity_rules).any(axis=1).mean()
print(f"Integrity Violation Rate: {integrity_violation_rate:.2%}")

Integrity Violation Rate: 18.18%


In [13]:
timeliness_rate = (~timeliness_mask & valid_dates).sum() / valid_dates.sum()

print(f"Timeliness Rate (<=7 days): {timeliness_rate:.2%}")

Timeliness Rate (<=7 days): 66.67%


In [14]:
# Flagged cells across all dimensions combined
field_errors = (
        df.isna().sum().sum() +
        dup_mask.sum() +
        pd.DataFrame(validation_rules).sum().sum() +
        pd.DataFrame(consistency_rules).sum().sum() +
        pd.DataFrame(integrity_rules).sum().sum() +
        timeliness_mask.sum()
)

total_cells = len(df) * len(df.columns)
field_error_rate = field_errors / total_cells

print(f"Field Errors: {field_errors}")
print(f"Field Error Rate: {field_error_rate:.2%}")

Field Errors: 20
Field Error Rate: 20.20%


In [15]:
# Record-level error rate: records with at least one violation across all dimensions
all_violations = pd.DataFrame({
    **validation_rules,
    **consistency_rules,
    **integrity_rules,
    'missing_value': df.isna().any(axis=1),
    'duplicate': dup_mask,
    'timeliness': timeliness_mask,
})

record_error_mask = all_violations.any(axis=1)
error_rate = record_error_mask.mean()

print(f"Error Rate: {error_rate:.2%}")
print(f"\nRecords with at least one violation:")
df[record_error_mask]

Error Rate: 90.91%

Records with at least one violation:


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
7,T0007,C106,2026-02-30,47.00,EUR,card,completed,DE,2026-02-15
8,T0008,C107,2026-01-10,1000000.00,EUR,card,completed,DE,2026-01-10
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN
10,T0010,C109,2026-01-11,52.10,EUR,NaN,completed,NL,2026-01-11


In [16]:
# All KPIs normalised to higher is better before averaging
# All five dimensions are weighted equally in the overall score. In a real-world scenario, weights would be assigned based on business impact.

overall = (
                  completeness_rate +
                  (1 - duplication_rate) +
                  (1 - validity_error_rate) +
                  (1 - integrity_violation_rate) +
                  timeliness_rate
          ) / 5

print(f"Overall Data Quality Score: {overall:.2%}")

Overall Data Quality Score: 76.16%


Five dimension-level KPIs were computed directly from the violation masks defined in
Task 2, alongside a record-level error rate, a field error rate, and an overall quality
score. All KPIs were normalised to higher-is-better before being averaged into the
overall score, meaning rates where lower values indicate better quality (duplication,
validity error, integrity violation) were inverted as `(1 - rate)` before averaging.
All five dimensions are weighted equally. In a real-world scenario, weights would be
assigned based on business impact.

The completeness rate of 95.96% indicates that four cells are missing across the dataset.
The duplication rate of 18.18% is caused by the fully duplicated T0006 record.
The validity error rate of 45.45% reflects five records carrying at least one validity
violation across `transaction_date`, `amount`, and `currency`. The integrity violation
rate of 18.18% corresponds to two records with referential or temporal violations. The
timeliness rate of 66.67% indicates that six out of nine records with valid dates were
updated within the seven-day threshold.

The record-level error rate of 90.91% reflects that 10 out of 11 records carry at least
one violation when all dimensions are considered together. The field error rate of 20.20%
counts 20 individually flagged cells out of 99 total, giving a cell-level view that
complements the record-level rate. The overall quality score of 76.16% is an equally
weighted composite of the five dimension-level KPIs.

# Task 4: Audit Summary
Prepare a short audit summary with issue type, affected rows, severity, and recommended next action.

Severity ratings below are assigned based on the nature of the issue within this dataset.
In practice, severity depends on the specific business context, regulatory environment,
and downstream systems, which are not known here.

| Issue                                | Dimension               | Affected Rows | Severity | Severity Justification                                                                                                                                                             | Recommended Next Action                                                                                                           |
|--------------------------------------|-------------------------|---------------|----------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------|
| Duplicate transaction_id             | Uniqueness              | 2             | High     | transaction_id is the primary key; duplicate values mean two records are indistinguishable                                                                                         | Trace root cause in key-generation process; establish deduplication policy                                                        |
| Missing customer_id                  | Completeness, Integrity | 1             | High     | customer_id is the primary reference linking a transaction to a customer; missing value means no customer attribution and the record cannot participate in any customer-level join | Escalate to source system owner; quarantine record from downstream joins until resolved; enforce non-null constraint at ingestion |
| Missing amount                       | Completeness            | 1             | High     | amount is the primary numeric measure of the transaction; missing value means the record cannot contribute to any financial aggregation                                            | Investigate source system for incomplete writes; enforce non-null constraint at ingestion                                         |
| Impossible date 2026-02-30           | Validity                | 1             | High     | The date does not exist in the calendar; the record cannot be placed on a timeline                                                                                                 | Reject at ingestion; enforce calendar validation at entry point                                                                   |
| Negative amount                      | Validity                | 1             | High     | Value falls outside the valid range for a transaction amount                                                                                                                       | Investigate transaction processing logs; introduce amount range validation at ingestion                                           |
| Outlier amount 1,000,000             | Validity                | 1             | High     | Value is three orders of magnitude above the next highest amount in the dataset                                                                                                    | Escalate for manual review; introduce amount ceiling validation                                                                   |
| last_updated before transaction_date | Integrity               | 1             | High     | last_updated precedes transaction_date, which violates the temporal relationship between the two fields                                                                            | Investigate pipeline timestamps; audit upstream system clocks                                                                     |
| Missing payment_method               | Completeness            | 1             | Medium   | payment_method is a required attribute for transaction classification; missing value means the record cannot be categorised by payment method                                      | Investigate source system; enforce non-null constraint at ingestion                                                               |
| Zero amount                          | Validity                | 1             | Medium   | Value falls outside the valid range for a transaction amount                                                                                                                       | Flag for business review; introduce non-zero constraint for completed transactions                                                |
| Non-standard currency EURO           | Validity                | 1             | Medium   | EURO is not a recognised ISO 4217 currency code; the correct code is EUR                                                                                                           | Enforce currency whitelist at ingestion; map EURO to EUR                                                                          |
| Non-standard date format 06-01-2026  | Consistency             | 1             | Medium   | Format is ambiguous; day and month cannot be determined without additional context                                                                                                 | Enforce ISO 8601 format at entry point                                                                                            |
| Update lag exceeds 7 days            | Timeliness              | 3             | Medium   | Three records have a gap greater than seven days between transaction_date and last_updated                                                                                         | Investigate update process; set automated alert for lag exceeding threshold                                                       |
| Missing last_updated                 | Completeness            | 1             | Low      | One record has no last_updated value; the update history for that record cannot be determined                                                                                      | Flag for monitoring; enforce at ingestion in next pipeline version                                                                |
| Non-standard date format 2026/01/06  | Consistency             | 1             | Low      | Format deviates from ISO 8601 but is unambiguous                                                                                                                                   | Standardise to YYYY-MM-DD at entry point                                                                                          |
| Mixed case CARD                      | Consistency             | 1             | Low      | Value does not match the lowercase standard applied to payment_method                                                                                                              | Normalise to lowercase at ingestion                                                                                               |
| Mixed case region de                 | Consistency             | 1             | Low      | Value does not match the uppercase standard applied to region                                                                                                                      | Normalise to uppercase at ingestion                                                                                               |

# Task 5: Suggestions
Suggest suitable cleaning steps for the next chapter, but do not implement full cleaning yet.

The following cleaning steps apply only to the existing dataset and do not involve
reacquiring or replacing source data.

Duplicate records flagged on `transaction_id` should be investigated to determine which
copy is correct before any record is removed. Date formatting inconsistencies can be
resolved by standardising to ISO 8601 using `pd.to_datetime` with `errors='coerce'`,
which handles non-standard formats and coerces impossible dates such as 2026-02-30 to
NaN. Categorical inconsistencies in `currency`, `payment_method`, and `region` can be
normalised using `.str.upper()`, `.str.lower()`, and a targeted `.replace()` for the
EURO case. A derived column capturing the lag in days between `transaction_date` and
`last_updated` should be retained for downstream monitoring and timeliness reporting.

Several issues identified in Task 4 fall outside the scope of programmatic cleaning.
The negative amount, missing `customer_id`, missing `amount`, missing `payment_method`,
missing `last_updated`, the outlier of 1,000,000, and the `last_updated` before
`transaction_date` case all require investigation at the source system or a
business-level decision before any code should act on them. These records should be
quarantined and escalated before the cleaning chapter proceeds.